# Fine level clustering and annotation for CSF/blood validation dataset

In [1]:
suppressPackageStartupMessages({
  library(Seurat)
  library(future)
  library(tidyverse)
})

In [ ]:
options(future.globals.maxSize = 1000000 * 1024^2, hpc.ncpus = 16)
plan(multicore, workers = getOption("hpc.ncpus", 1))

In [ ]:
results.dir <- "../results/tables/external/cantoni/cluster_annotate"
dir.create(file.path(results.dir, "subcluster"), recursive = TRUE)

# Helper functions

In [3]:
# Modified implementation of Seurat:::RunLeiden() using leidenbase::leiden_find_partition() instead of leiden::leiden() for speed
RunLeiden <- function(object,
                      partition.type = c(
                        "RBConfigurationVertexPartition",
                        "ModularityVertexPartition",
                        "RBERVertexPartition",
                        "CPMVertexPartition",
                        "SignificanceVertexPartition",
                        "SurpriseVertexPartition"
                      ),
                      initial.membership = NULL,
                      node.sizes = NULL,
                      resolution.parameter = 1,
                      random.seed = 0,
                      n.iter = 10,
                      ...) {
  input <- if (inherits(x = object, what = "list")) {
    igraph::graph_from_adj_list(adjlist = object)
  } else if (inherits(x = object, what = c("dgCMatrix", "matrix", "Matrix"))) {
    if (inherits(x = object, what = "Graph")) {
      object <- Seurat::as.sparse(x = object)
    }
    igraph::graph_from_adjacency_matrix(adjmatrix = object, weighted = TRUE)
  } else if (inherits(x = object, what = "igraph")) {
    object
  } else {
    stop("Input object must be a list, matrix, dgCMatrix, Matrix, or igraph object.")
  }

  partition <- leidenbase::leiden_find_partition(
    igraph = input,
    partition_type = partition.type,
    initial_membership = initial.membership,
    edge_weights = NULL,
    node_sizes = node.sizes,
    resolution_parameter = resolution.parameter,
    seed = ifelse(random.seed < 1, 1, random.seed),
    num_iter = n.iter
  )
  return(partition[[1]])
}

assignInNamespace("RunLeiden", RunLeiden, ns = "Seurat")

# Load data

### Full dataset

QS object generated by running notebooks in `02_prepare_external_datasets`

In [413]:
seu <- qs::qread(file = "../data/processed/external/cantoni/annotated/cantoni_untreated_ms_hc.qs", nthreads = getOption("hpc.ncpus", 1))

In [414]:
seu

An object of class Seurat 
18841 features across 184296 samples within 1 assay 
Active assay: RNA (18841 features, 0 variable features)
 3 layers present: counts, scale.data, data
 2 dimensional reductions calculated: pca, integrated.rna

### Cluster annotation

Cluster annotation TSV generated by running `slurm/reference_map_cantoni.sh`

In [ ]:
cluster.annotation <- read.table(
  file = file.path(results.dir, "reference_map.tsv"),
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  select(cluster_fine) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_macrophage", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        c("B_naive_transitional", "B_naive") ~ "B_naive",
        "B_memory" ~ "B_memory",
        "B_plasma" ~ "B_plasma",
        "DC_AXL_SIGLEC6" ~ "DC_AXL_SIGLEC6",
        c("DC_conventional_1", "DC_conventional_2") ~ "DC_conventional",
        "DC_plasmacytoid" ~ "DC_plasmacytoid",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16") ~ "Mono_CD14",
        c("Mono_CD16", "Mono_CD16_IFN") ~ "Mono_CD16",
        "NK_CD56bright" ~ "NK_CD56bright",
        "NK_CD56dim" ~ "NK_CD56dim",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN") ~ "T_CD4_naive",
        c("T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector") ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        c("T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN") ~ "T_CD8_naive",
        c("T_CD8_memory_central", "T_CD8_memory_effector") ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        c("B_naive", "B_memory", "B_plasma") ~ "B",
        c("DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid") ~ "DC",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD16") ~ "Mono",
        c("NK_CD56bright", "NK_CD56dim") ~ "NK",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_memory", "T_regulatory_naive", "T_regulatory_memory",
          "T_CD8_naive", "T_CD8_memory", "T_MAIT", "T_GD", "T_DN") ~ "T"
      ),
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

## B cells

In [416]:
# Subset to B cells only
sub <- subset(seu, subset = cluster_main == "B")
Idents(sub) <- "cluster_fine"

### Subcluster naive B cells

In [35]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "B_naive") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ "B_memory",
  "6" ~ "Artefact"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster memory B cells

In [45]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "B_memory" | cluster_sub == "B_memory") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ "Artefact"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster plasma cells

In [53]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "B_plasma") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ "B_plasma",
  "2" ~ "B_plasma",
  "3" ~ "B_plasma",
  "4" ~ "Proliferating"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Merge subclusters with fine and coarse cluster annotations

In [54]:
sub[[]] <- sub[[]] %>%
  mutate(
    cluster_fine = factor(
      if_else(is.na(cluster_sub), cluster_fine, cluster_sub),
      levels = c(
        "Artefact",
        "B_naive_transitional",
        "B_naive",
        "B_memory",
        "B_plasma",
        "Proliferating"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        "B_naive_transitional" ~ "B_naive",
        "B_naive" ~ "B_naive",
        "B_memory" ~ "B_memory",
        "B_plasma" ~ "B_plasma",
        "Proliferating" ~ "Proliferating"
      ),
      levels = c("Artefact", "B_naive", "B_memory", "B_plasma", "Proliferating")
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        "B_naive" ~ "B",
        "B_memory" ~ "B",
        "B_plasma" ~ "B",
        "Proliferating" ~ "Proliferating"
      ),
      levels = c("Artefact", "B", "Proliferating")
    )
  )
Idents(sub) <- "cluster_fine"

In [ ]:
write.table(
  sub[[c("cluster_main", "cluster_coarse", "cluster_fine")]],
  file = file.path(results.dir, "subcluster/subcluster_annotation_b.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

## Monocytes

In [56]:
# Subset to monocytes only
sub <- subset(seu, subset = cluster_main == "Mono")
Idents(sub) <- "cluster_fine"

### Subcluster CD14 monocytes

In [116]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_coarse == "Mono_CD14") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 1,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ NA,
  "7" ~ NA,
  "8" ~ "Mono_CD14_macrophage",
  "9" ~ "Mono_CD14_IFN",
  "10" ~ "Doublet",
  "11" ~ "Artefact",
  "12" ~ "Artefact"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster CD16 monocytes

In [117]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_coarse == "Mono_CD16") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ "Mono_CD14_macrophage",
  "5" ~ "Mono_CD16_IFN",
  "6" ~ "Doublet"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Merge subclusters with fine and coarse cluster annotations

In [119]:
sub[[]] <- sub[[]] %>%
  mutate(
    cluster_fine = factor(
      if_else(is.na(cluster_sub), cluster_fine, cluster_sub),
      levels = c(
        "Artefact",
        "Doublet",
        "Mono_CD14",
        "Mono_CD14_platelet",
        "Mono_CD14_IL1B",
        "Mono_CD14_IFN",
        "Mono_CD14_macrophage",
        "Mono_CD14_CD16",
        "Mono_CD16",
        "Mono_CD16_IFN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        "Doublet" ~ "Doublet",
        "Mono_CD14" ~ "Mono_CD14",
        "Mono_CD14_platelet" ~ "Mono_CD14",
        "Mono_CD14_IL1B" ~ "Mono_CD14",
        "Mono_CD14_IFN" ~ "Mono_CD14",
        "Mono_CD14_macrophage" ~ "Mono_CD14",
        "Mono_CD14_CD16" ~ "Mono_CD14",
        "Mono_CD16" ~ "Mono_CD16",
        "Mono_CD16_IFN" ~ "Mono_CD16"
      ),
      levels = c(
        "Artefact",
        "Doublet",
        "Mono_CD14",
        "Mono_CD16"
      ),
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        "Doublet" ~ "Doublet",
        "Mono_CD14" ~ "Mono",
        "Mono_CD16" ~ "Mono"
      ),
      levels = c("Artefact", "Doublet", "Mono")
    )
  )
Idents(sub) <- "cluster_fine"

In [ ]:
write.table(
  sub[[c("cluster_main", "cluster_coarse", "cluster_fine")]],
  file = file.path(results.dir, "subcluster/subcluster_annotation_mono.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

## NK cells

In [121]:
# Subset to NK cells only
sub <- subset(seu, subset = cluster_main == "NK")
Idents(sub) <- "cluster_fine"

### Subcluster NK cells

In [146]:
plan(sequential)
tmp <- sub %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 2,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ "NK_CD56bright",
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ NA,
  "7" ~ NA,
  "8" ~ NA,
  "9" ~ NA,
  "10" ~ NA,
  "11" ~ NA,
  "12" ~ "T_CD8_memory_effector",
  "13" ~ NA,
  "14" ~ NA,
  "15" ~ NA,
  "16" ~ "Proliferating",
  "17" ~ "Doublet",
  "18" ~ "Artefact",
  "19" ~ NA
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph



Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Merge subclusters with fine and coarse cluster annotations

In [148]:
sub[[]] <- sub[[]] %>%
  mutate(
    cluster_fine = factor(
      if_else(is.na(cluster_sub), cluster_fine, cluster_sub),
      levels = c(
        "Artefact",
        "Doublet",
        "NK_CD56bright",
        "NK_CD56dim",
        "Proliferating",
        "T_CD8_memory_effector"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        "Doublet" ~ "Doublet",
        "NK_CD56bright" ~ "NK_CD56bright",
        "NK_CD56dim" ~ "NK_CD56dim",
        "Proliferating" ~ "Proliferating",
        "T_CD8_memory_effector" ~ "T_CD8_memory",
      ),
      levels = c(
        "Artefact",
        "Doublet",
        "NK_CD56bright",
        "NK_CD56dim",
        "Proliferating",
        "T_CD8_memory"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        "Doublet" ~ "Doublet",
        "NK_CD56bright" ~ "NK",
        "NK_CD56dim" ~ "NK",
        "Proliferating" ~ "Proliferating",
        "T_CD8_memory" ~ "T"
      ),
      levels = c("Artefact", "Doublet", "NK", "Proliferating", "T")
    )
  )
Idents(sub) <- "cluster_fine"

In [ ]:
write.table(
  sub[[c("cluster_main", "cluster_coarse", "cluster_fine")]],
  file = file.path(results.dir, "subcluster/subcluster_annotation_nk.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

## T cells

In [ ]:
# Add misclassified CD8 effectory memory T cells to T cells
subcluster.annotation.nk <- read.table(
  file = file.path(results.dir, "subcluster/subcluster_annotation_nk.tsv"),
  sep = "\t",
  header = TRUE,
  row.names = 1
)
seu <- AddMetaData(seu, metadata = subcluster.annotation.nk)
# Subset to T cells only
sub <- subset(seu, subset = cluster_main == "T")
Idents(sub) <- "cluster_fine"

### Subcluster naive CD8 T cells

In [167]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_coarse == "T_CD8_naive") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ "T_CD8_memory_central",
  "2" ~ "T_CD8_memory_central",
  "3" ~ NA,
  "4" ~ "T_CD8_memory_effector"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster naive CD4 T cells

In [182]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_coarse == "T_CD4_naive") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ "Mono_CD14_macrophage",
  "6" ~ "T_regulatory_naive",
  "7" ~ "Doublet",
  "8" ~ "Doublet",
  "9" ~ "T_CD4_naive_IFN"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster effector memory CD4 T cells

In [233]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "T_CD4_memory_effector") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 1,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ "T_CD4_memory_central",
  "2" ~ NA,
  "3" ~ "T_CD4_memory_central",
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ NA,
  "7" ~ NA,
  "8" ~ NA,
  "9" ~ NA,
  "10" ~ "T_regulatory_memory",
  "11" ~ NA,
  "12" ~ "T_CD4_memory_central_IFN",
  "13" ~ "Mono_CD14_macrophage",
  "14" ~ "Artefact"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster central memory CD4 T cells

In [266]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "T_CD4_memory_central" | cluster_sub == "T_CD4_memory_central") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 1,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ NA,
  "7" ~ NA,
  "8" ~ "T_regulatory_memory",
  "9" ~ NA,
  "10" ~ NA
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster effector memory CD8 T cells

In [327]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "T_CD8_memory_effector" | cluster_sub == "T_CD8_memory_effector") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 1.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ NA,
  "7" ~ NA,
  "8" ~ "T_CD4_memory_effector",
  "9" ~ NA,
  "10" ~ NA,
  "11" ~ "Artefact",
  "12" ~ NA,
  "13" ~ "T_CD4_memory_effector",
  "14" ~ NA,
  "15" ~ "Proliferating",
  "16" ~ "Doublet",
  "17" ~ "Artefact"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster central memory CD8 T cells

In [349]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "T_CD8_memory_central" | cluster_sub == "T_CD8_memory_central") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ "T_CD8_memory_effector",
  "3" ~ "Artefact",
  "4" ~ "T_CD8_memory_effector",
  "5" ~ "Artefact"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster gamma delta T cells

In [364]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "T_GD") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 1,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ "Artefact"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Subcluster MAIT cells

In [385]:
plan(sequential)
tmp <- sub %>%
  subset(subset = cluster_fine == "T_MAIT") %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 1,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ NA,
  "7" ~ "Doublet"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph

Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Merge subclusters with fine and coarse cluster annotations

In [387]:
sub[[]] <- sub[[]] %>%
  mutate(
    cluster_fine = factor(
      if_else(is.na(cluster_sub), cluster_fine, cluster_sub),
      levels = c(
        "Artefact",
        "Doublet",
        "Mono_CD14_macrophage",
        "Proliferating",
        "T_CD4_naive",
        "T_CD4_naive_SOX4",
        "T_CD4_naive_IFN",
        "T_CD4_memory_central",
        "T_CD4_memory_central_IFN",
        "T_CD4_memory_effector",
        "T_regulatory_naive",
        "T_regulatory_memory",
        "T_CD8_naive",
        "T_CD8_naive_SOX4",
        "T_CD8_naive_IFN",
        "T_CD8_memory_central",
        "T_CD8_memory_effector",
        "T_MAIT",
        "T_GD",
        "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        "Doublet" ~ "Doublet",
        "Mono_CD14_macrophage" ~ "Mono_CD14",
        "Proliferating" ~ "Proliferating",
        "T_CD4_naive" ~ "T_CD4_naive",
        "T_CD4_naive_SOX4" ~ "T_CD4_naive",
        "T_CD4_naive_IFN" ~ "T_CD4_naive",
        "T_CD4_memory_central" ~ "T_CD4_memory",
        "T_CD4_memory_central_IFN" ~ "T_CD4_memory",
        "T_CD4_memory_effector" ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        "T_CD8_naive" ~ "T_CD8_naive",
        "T_CD8_naive_SOX4" ~ "T_CD8_naive",
        "T_CD8_naive_IFN" ~ "T_CD8_naive",
        "T_CD8_memory_central" ~ "T_CD8_memory",
        "T_CD8_memory_effector" ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "Doublet",
        "Mono_CD14",
        "Proliferating",
        "T_CD4_naive",
        "T_CD4_memory",
        "T_regulatory_naive",
        "T_regulatory_memory",
        "T_CD8_naive",
        "T_CD8_memory",
        "T_MAIT",
        "T_GD",
        "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        "Doublet" ~ "Doublet",
        "Mono_CD14" ~ "Mono",
        "Proliferating" ~ "Proliferating",
        "T_CD4_naive" ~ "T",
        "T_CD4_memory" ~ "T",
        "T_regulatory_naive" ~ "T",
        "T_regulatory_memory" ~ "T",
        "T_CD8_naive" ~ "T",
        "T_CD8_memory" ~ "T",
        "T_MAIT" ~ "T",
        "T_GD" ~ "T",
        "T_DN" ~ "T"
      ),
      levels = c("Artefact", "Doublet", "Mono", "Proliferating", "T")
    )
  )
Idents(sub) <- "cluster_fine"

In [ ]:
write.table(
  sub[[c("cluster_main", "cluster_coarse", "cluster_fine")]],
  file = file.path(results.dir, "subcluster/subcluster_annotation_t.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

## Dendritic cells

In [389]:
# Subset to dendritic cells only
sub <- subset(seu, subset = cluster_main == "DC")
Idents(sub) <- "cluster_fine"

### Subcluster dendritic cells

In [404]:
plan(sequential)
tmp <- sub %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 0.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ "Mono_CD14_macrophage"
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph



Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Merge subclusters with fine and coarse cluster annotations

In [405]:
sub[[]] <- sub[[]] %>%
  mutate(
    cluster_fine = factor(
      if_else(is.na(cluster_sub), cluster_fine, cluster_sub),
      levels = c(
        "Mono_CD14_macrophage",
        "DC_AXL_SIGLEC6",
        "DC_conventional_1",
        "DC_conventional_2",
        "DC_plasmacytoid"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Mono_CD14_macrophage" ~ "Mono_CD14",
        "DC_AXL_SIGLEC6" ~ "DC_AXL_SIGLEC6",
        "DC_conventional_1" ~ "DC_conventional",
        "DC_conventional_2" ~ "DC_conventional",
        "DC_plasmacytoid" ~ "DC_plasmacytoid"
      ),
      levels = c(
        "Mono_CD14",
        "DC_AXL_SIGLEC6",
        "DC_conventional",
        "DC_plasmacytoid"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Mono_CD14" ~ "Mono",
        "DC_AXL_SIGLEC6" ~ "DC",
        "DC_conventional" ~ "DC",
        "DC_plasmacytoid" ~ "DC"
      ),
      levels = c("Mono", "DC")
    )
  )
Idents(sub) <- "cluster_fine"

In [ ]:
write.table(
  sub[[c("cluster_main", "cluster_coarse", "cluster_fine")]],
  file = file.path(results.dir, "subcluster/subcluster_annotation_dc.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

# Update cluster annotation

In [ ]:
subcluster.annotation <- lapply(
  c("B", "DC", "Mono", "NK", "T"), # order is important (cells present in later annotation files will overwrite earlier ones)
  function(x) {
    read.table(
      file = glue::glue("{results.dir}/subcluster/subcluster_annotation_{tolower(x)}.tsv"),
      sep = "\t",
      header = TRUE,
      row.names = 1,
      stringsAsFactors = FALSE
    )
  }
) %>%
  Reduce(
    f = function(x, y) {
      filtered <- x[!rownames(x) %in% rownames(y), , drop = FALSE]
      result <- rbind(filtered, y)
      return(result)
    },
    x = .
  )

In [418]:
cluster.annotation[rownames(subcluster.annotation), ] <- subcluster.annotation[, colnames(cluster.annotation)]
cluster.annotation <- cluster.annotation %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_macrophage","Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      cluster_coarse,
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      cluster_main,
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

# Save final cluster annotations

In [ ]:
write.table(
  cluster.annotation,
  file = file.path(results.dir, "cluster_annotation_final.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

# Session info

In [40]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /ceph/project/fuggerlab/rfarooq/.conda/envs/sandbox/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
[1] C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] reticulate_1.39.0  lubridate_1.9.3    forcats_1.0.0      stringr_1.5.1     
 [5] dplyr_1.1.4        purrr_1.0.2        readr_2.1.5        tidyr_1.3.1       
 [9] tibble_3.2.1       ggplot2_3.5.1      tidyverse_2.0.0    future_1.34.0     
[13] Seurat_5.1.0       SeuratObject_5.0.2 sp_2.1-4          

loaded via a namespace (and not attached):
  [1] RColorBrewer_1.1-3     jsonlite_1.8.9         magrittr_2.0.3        
  [4] spatstat.utils_3.1-0   farver_2.1.2           vctrs_0.6.5           
  [7] ROCR_1.0-11            spatstat.explore_3.2-6 base6